# Current Segment and Text Diagnostics

**Current dataset notebook.** This notebook has been refreshed for the enriched current BOAMP dataset under `data/processed/boamp_current/`.

Uses the current enriched BOAMP dataset to inspect segment distributions and text fields; older NLP annotation outputs are historical.

Interpretation guardrail: `event = 1` is a **proxy recurrence**, meaning an identifiable reappearance of a similar procurement need under the selected rule. It is not a legally verified renewal.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd()
PROJECT = cwd
for candidate in [cwd, *cwd.parents]:
    if (candidate / "data").exists() and (candidate / "reports").exists():
        PROJECT = candidate
        break

DATA = PROJECT / "data" / "processed" / "boamp_current"
RAW = PROJECT / "data" / "raw" / "boamp_current"
TABLES = PROJECT / "reports" / "tables"
FIGURES = PROJECT / "reports" / "figures"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {PROJECT}")
print(f"Current data directory: {DATA}")


## Current BOAMP Download and Dataset Shape


In [ ]:
download = pd.read_csv(TABLES / "data" / "boamp_current_download_summary.csv")
enrichment = pd.read_csv(TABLES / "data" / "buyer_enrichment_summary.csv")
population_summary = pd.read_csv(TABLES / "data" / "analytical_population_summary.csv")
selected = pd.read_csv(TABLES / "linkage" / "final_selected_event_definition_current.csv")
survival = pd.read_csv(TABLES / "survival" / "survival_summary_current.csv")
risk = pd.read_csv(TABLES / "survival" / "operational_risk_scores_current.csv", dtype={"SIREN": str, "SIRET": str})

summary = {
    "study_period": download["actual_date_range"].dropna().iloc[-1],
    "retained_notices": int(download["number_of_retained_notices"].dropna().iloc[-1]),
    "appel_offre": int(enrichment.loc[enrichment["metric"].eq("APPEL_OFFRE"), "value"].iloc[0]),
    "eligible_contracts": int(selected["eligible_contracts"].iloc[0]),
    "selected_method": f"{selected['selected_method'].iloc[0]} {selected['selected_variant'].iloc[0]}",
    "proxy_events": int(selected["event_count"].iloc[0]),
    "event_rate": float(selected["event_rate"].iloc[0]),
    "censoring_date": selected["censoring_date"].iloc[0],
    "survival_24m": float(survival["survival_24m"].iloc[0]),
    "mean_p12": float(risk["p_renewal_12m"].mean()),
    "mean_p24": float(risk["p_renewal_24m"].mean()),
}
pd.DataFrame([summary])


In [ ]:
clean = pd.read_csv(DATA / "boamp_full_clean_enriched.csv", dtype={"buyer_siret_clean": str, "buyer_siren_enriched": str}, low_memory=False)
download_by_year = pd.read_csv(TABLES / "data" / "boamp_current_download_summary.csv")
display(download_by_year[["year", "retained_notices", "api_total_count", "pages"]])
display(clean[["idweb", "dateparution", "nature", "buyer_key", "buyer_key_type", "cpv_clean", "category_label"]].head())


## Buyer Enrichment and Data Quality


In [ ]:
display(pd.read_csv(TABLES / "data" / "buyer_enrichment_summary.csv"))
display(pd.read_csv(TABLES / "data" / "buyer_key_quality_summary.csv"))
display(pd.read_csv(TABLES / "data" / "siren_siret_coverage_by_year.csv").tail())
display(pd.read_csv(TABLES / "data" / "cpv_quality_by_year.csv").tail())
display(pd.read_csv(TABLES / "data" / "duration_quality_by_year.csv").tail())


## Current Figures


In [ ]:
from IPython.display import Image, display

for fig in [
    FIGURES / "data" / "analytical_population_funnel.png",
    FIGURES / "linkage" / "method_event_counts.png",
    FIGURES / "linkage" / "method_score_distributions.png",
    FIGURES / "survival" / "km_curve_current.png",
    FIGURES / "survival" / "p12_distribution_current.png",
]:
    print(fig.relative_to(PROJECT))
    if fig.exists():
        display(Image(filename=str(fig)))
    else:
        print("MISSING")
